#Basic

In [0]:
difference between warehouse, lake and lakehouse is that the warehouse is a place where you can store data in structure format and in lake you can store data in unstructured format but it does not support ACID and lakehouse is a combination of warehouse and lake which means it can store both the structured as well as unstructured data and it supports ACID

In [0]:
%sql
create or replace table dev.bronze.products (product_id int, name string, category string, price double) using delta

In [0]:
%sql
insert into dev.bronze.products(values (1, 'iPhone', 'electronics', 999.99), (2, 'MacBook', 'electronics', 1999.99), (3, 'T-Shirt', 'clothing', 19.99), (4, 'Pants', 'clothing', 29.99), (5, 'Shoes', 'shoes', 49.99), (6, 'Socks', 'shoes', 9.99), (7, 'Hat', 'accessories', 14.99), (8, 'Watch', 'accessories', 149.99), (9, 'Jacket', 'clothing', 49.99), (10, 'Sunglasses', 'accessories', 39.99))

In [0]:
%sql
describe history dev.bronze.products

In [0]:
%sql
update dev.bronze.products set name = 'iPhone 13' where product_id = 1

In [0]:
%sql
select * from dev.bronze.products version as of 1

In [0]:
%sql
select * from dev.bronze.products version as of 2

--note : the version as of 1 is different from the current version as of 2 as it updated the name of the product 1
-- both version have differnet name for product 1 for version 1 its contains name before update and for the version 2 its contains the updated name

#Intermediate

In [0]:
%sql
insert into dev.bronze.products(values (11, 'Laptop', 'electronics', 1299.99, '2023-01-01'))

In [0]:
data = [(11, 'Laptop', 'electronics', 1299.99, '2023-01-01')]
schema = ["product_id", "name", "category", "price", "date"]
df = spark.createDataFrame(data, schema = schema)
df.write.option("mergeSchema", "true").mode("append").saveAsTable("dev.bronze.products")

In [0]:
%sql
restore table dev.bronze.products version as of 2

In [0]:
%sql
select * from dev.bronze.products

In [0]:
ACID transactions matter because they prevent data corruption and conditions
suppose if their are two pipelines A and B and both are reading the same table and if pipeline B made some changes on the table and if pipeline A is reading the same table then it will not get the latest changes made by pipeline B until and unless the pipeline B commits the changes this is because of ACID transactions through isolation and atomacity

#Advanced

In [0]:
load data continuosly in small batches using delta lake instead of using one nightly job
its safer as it uses ACID which will load the whole data there is no half done load so their are low chances of job crashes people can query it anytime 
also is uses rollback which means we can also have access to the old data 

In [0]:
data = [(11, "iphone 13", "electonics", 1999.99)]
df = spark.createDataFrame(data, ["product_id", "name", "category", "price"])
df.write.format("delta").mode("append").saveAsTable("dev.bronze.products")

In [0]:
data = [(12, "iphone 14", "electonics", 2999.99)]
df = spark.createDataFrame(data, ["product_id", "name", "category", "price"])
df.write.format("delta").mode("append").saveAsTable("dev.bronze.products")

In [0]:
%sql
desc history dev.bronze.products

--no the write does not conflict here as it is creating the new parquet file and not overwriting the existing one because the write operation is append which always create new file does not look at the old files 
--both the write operations will appear as different version

In [0]:
Advantages of lakehouse over traditional warehouse

1. lakehouse gives fresh data because here data lands continuously which provide fresh data always
2. as it support ACID either all the transaction will be commited or no one becasue of that we can run reports even when the new data is loading as it does not have any partial or broken data
3. it provides rollback if their is any problem we can check them by time travelling through version as of or timestamp as of

tradeof 
be mindful of doing updates in batches as it can be expensive for small or frequent updates and can use more storage than a traditional warehouse for the same task